# SparkClient: Python Function (FuncJob) Lifecycle

This notebook demonstrates how to submit and manage an in-memory Python function as a distributed Spark job using `FuncJob` and `SparkClient` from the Kubeflow SDK.

### What you will learn:
1. Defining a native Python function for execution on Spark.
2. Submitting the function via `FuncJob` with custom arguments and executor resources.
3. Polling and waiting for the job to complete.
4. Inspecting job metadata and driver pod information.
5. Listing active and completed jobs.
6. Streaming driver pod logs to observe function output.
7. Deleting the Spark job resource after execution.

## 1. Imports and Client Setup

Import necessary classes from `kubeflow.spark` and initialize `SparkClient` targeting your cluster namespace.

In [ ]:
import os

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import FuncJob, SparkClient, SparkJobStatus

# Namespace configuration (uses SPARK_TEST_NAMESPACE if set, else 'default')
namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)

# Initialize SparkClient
client = SparkClient(backend_config=backend_config)
print(f"SparkClient initialized for namespace: {namespace}")

## 2. Define the Spark Function

Define the Python function to be executed remotely. Inside the function, initialize a `SparkSession`, perform computations, and terminate the session.

In [ ]:
def estimate_pi(samples: int) -> None:
    """Simple function executed as a Spark FuncJob."""
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("estimate-pi").getOrCreate()
    count = spark.range(samples).count()
    print(f"Estimated Pi over {count} samples.")
    spark.stop()


print("Spark function 'estimate_pi' defined successfully.")

## 3. Submit the FuncJob

Package the function into a `FuncJob` and submit it with executor configuration.

In [ ]:
print("Submitting Spark FuncJob...")
job_name = client.submit_job(
    job=FuncJob(
        func=estimate_pi,
        func_args={
            "samples": 10,
        },
    ),
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512Mi",
    },
)

print(f"Job submitted successfully: {job_name}")

## 4. Wait for Completion

Block until the job reaches completion or the timeout expires.

In [ ]:
print(f"Waiting for {job_name} to complete...")
job = client.wait_for_job_status(
    job_name,
    timeout=300,
)

print("Job completed successfully.")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Namespace: {job.namespace}")

## 5. Retrieve Job Metadata

Inspect job details returned by the Kubernetes Spark Operator.

In [ ]:
job_info = client.get_job(job_name)

print(f"Name: {job_info.name}")
print(f"Namespace: {job_info.namespace}")
print(f"Status: {job_info.status}")
print(f"Driver Pod: {job_info.driver_pod_name}")
print(f"Executors: {job_info.num_executors}")

## 6. List Spark Jobs

List all jobs in the namespace and verify completed status filtering.

In [ ]:
jobs = client.list_jobs()
print(f"Found {len(jobs)} total Spark job(s).")
for j in jobs:
    print(f"- {j.name} | Status: {j.status} | Namespace: {j.namespace}")

completed_jobs = client.list_jobs(status={SparkJobStatus.COMPLETED})
print(f"\nFound {len(completed_jobs)} completed Spark job(s).")

## 7. Stream Driver Logs

Inspect the first 20 lines of the driver pod logs to confirm the output of `estimate_pi`.

In [ ]:
print(f"Retrieving logs for {job_name} (first 20 lines):")
print("-" * 70)

for idx, line in enumerate(client.get_job_logs(job_name)):
    print(line.rstrip())
    if idx >= 20:
        print("...")
        break
print("-" * 70)

## 8. Clean Up Resources

Delete the completed Spark application resource.

In [ ]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Job deletion requested successfully.")